# Summary

Create a dataset with questions and queries

In [1]:
import os, sys
import json

import pandas as pd


utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()


## Get the log and questions files from GCP

In [2]:
# Get a list of contents in the GCP bucket that will be used for corpus
gcs_bucket_name = "c3pa-app"
gcs_path = "testing"
bcontents = gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                gcs_bucket_name=gcs_bucket_name,
                                path=gcs_path)
# print(bcontents)



# Read questions into a dataframe - It does seem this is needed since all questions are in logs


In [45]:
# # Read each question file
# questions_list = []
# for bc in bcontents:
#
#     if bc.find("questions") >= 0 and bc.find(".txt") >= 0:
#         filename = os.path.split(bc)[1]
#
#         ques_str = gct.read_gcs_text_file(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
#                                           gcs_bucket_name=gcs_bucket_name,
#                                           path=gcs_path,
#                                           filename=filename)
#
#         questions_list.extend(ques_str.split("\n"))
#
# questions_list = [q.strip() for q in questions_list if len(q) > 0]
# questions_list = set(questions_list)
# questions_list


## Read the logs into a dataframe

In [3]:
# Read each question file
logs_list = []
for bc in bcontents:

    if bc.find("log_testing") >= 0:
        filename = os.path.split(bc)[1]

        log_json = gct.read_json_file(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                      gcs_bucket_name=gcs_bucket_name,
                                      gcs_directory=gcs_path,
                                      file_name=filename,
                                      exact=False)

        logs_list.append(pd.DataFrame(data=log_json))

# Combine into a single dataframe
logs_df = pd.concat(objs=logs_list, axis=0, ignore_index=True)

# Drop duplicate questions
logs_df = logs_df.drop_duplicates(subset="question", keep="first")

logs_df.head()


,n,qn,question,prompt,response,start,stop,hash,version,user_input
0,1,10,What is the California community college with ...,What is the California community college with ...,The California Community College with the larg...,2025-02-26_18:14:07.442891,2025-02-26_18:14:33.797399,c032e4e4352ed9831156e739f22f49391d9d8162c54542...,25.01.30,NaN
1,2,10,How many colleges are there in the California ...,How many colleges are there in the California ...,The California Community College system is com...,2025-02-26_18:14:33.797601,2025-02-26_18:14:51.806109,77998fb0fbca6862faff2e6874007e088172268c71c0e2...,25.01.30,NaN
2,3,10,Do any California community colleges have a su...,Do any California community colleges have a su...,"Yes, several California community colleges are...",2025-02-26_18:14:51.806763,2025-02-26_18:15:09.986800,9c6dbfe6a76378c9d056d4a260b7db7e56f334ea1ec3ad...,25.01.30,NaN
3,4,10,How many California community colleges provide...,How many California community colleges provide...,While the California Community College system ...,2025-02-26_18:15:09.987020,2025-02-26_18:15:32.389602,91cead7ef30d1388d9f24f9bcd8eafe8e2190b6f302d9f...,25.01.30,NaN
4,5,10,What college is designated a Center of Excelle...,What college is designated a Center of Excelle...,Within the California Community College system...,2025-02-26_18:15:32.389785,2025-02-26_18:15:50.377177,d76cf722d4f1975b5ae4a1eaf7166591447802448aef27...,25.01.30,NaN


## Read the current Q A JSON

In [4]:
qa_adk_data_path = "eval/data"
qa_data_path = "../evaluation/rag_tests/data/eval_data"
# os.listdir(qa_data_path)

qa_file = "ipeds_qa_1.json"

# Read in q and a format
with open(os.path.join(qa_data_path, qa_file), "r") as file:
    qa_base = json.load(file)


In [39]:
# Save a prettier JSON
# qas_json = json.dumps(qa_base, indent=4)
# with open(os.path.join(qa_data_path, "ipeds_qa_2.json"), "w") as file:
#     file.write(qas_json)


## Get the corrected answers

In [5]:
corrected_answers = [
    ["What is the California community college with the largest student enrollment?",
     "According to Community College Review, the largest community college in California is East Los Angeles College with 48,505 students. The next largest California community college is East Los Angeles College with 48,505 students. Completing the 5 largest community colleges in California are American River College with 42,837 students, Bakersfield College with 39,530 and Santa Ana College with 38,105 students."],
    ["What is the annual budget for the California community colleges system?",
     "According to Legislative Analyst's office, Total CCC Funding Is $19 Billion in 2025‑26 Under Governor’s Budget. This reflects a 1.6 percent increase over the revised 2024‑25 level. $13.6 billion (72 percent) of CCC support in 2025‑26 would come from Proposition 98 funds. Proposition 98 funds, which consist of state General Fund and certain local property tax revenue, cover community colleges’ main operations. An additional $682 million non‑Proposition 98 General Fund would cover certain other costs, including debt service on state general obligation bonds for CCC facilities, a portion of CCC faculty retirement costs, and Chancellor’s Office operations. In recent years, the state has also provided non‑Proposition 98 General Fund for certain student housing projects."],
    ["How much does it cost to attend a California community college for one year?",
    "According to Community College Review, For California community colleges, the average tuition is approximately $1,236 per year for in-state students and $6,547 for out-of-state students (2025). The community college with the highest tuition in California is West Coast University-Los Angeles, with a tuition of $32,591. The community college with the lowest tuition is San Bernardino Valley College, with a tuition of $2,829."],
    ["Which California community college has a Culinary Arts Center?", "Southwestern College, Orange Coast College, Cuesta College and Santa Barbara City College all have programs in Culinary Arts. Orange Coast College, Riverside City College and Mt San Antonio College have programs in baking and pastry arts."],
    ["Which California community colleges have a student run bakery?", "Here are some California Community Colleges known to have student-run bakeries or cafes with significant baking components: San Joaquin Delta College (Stockton): Their Culinary Arts program features 'The Artisan Bakery,' run by Baking and Pastry students. They offer a variety of baked goods, pastries, and sandwiches to the campus community. You can find them on Instagram and Facebook as 'Artisan Bakery' and on TikTok as '@sjdc.artisanbakery'. Santa Rosa Junior College (Santa Rosa): The Culinary Arts program operates 'The Culinary Café & Bakery.' This student-run establishment is open Thursdays and Fridays when the college is in session and offers seasonal, mostly organic food. The bakery section features items prepared by students in the Baking & Pastry Certificate Program, often with gluten-free and vegan options available. Reservations for lunch at the café are recommended and can be made via their website. Long Beach City College (Long Beach): The Culinary Arts Department runs a 'Bakery & Bistro.' The Bakery is operated by Baking & Pastry Arts students and offers a range of sweet and savory items for take-out, such as croissants, tarts, brioche, and cookies. The menu includes staple items and weekly specials."],
    ["Which California community colleges have artist in residence programs?", "While specific, dedicated 'Artist in Residence' programs might not be a standard offering at all California Community Colleges, several colleges have strong arts programs and initiatives that could involve visiting artists or artist engagement in different capacities. Based on the search results, here are a few California Community Colleges that have demonstrably had artist-in-residence programs or related activities: Fullerton College: This college explicitly boasts a 'viable and prestigious Artist in Residence program' within its Art Department. They have a history of hosting national and international artists for residencies that typically include studio demonstrations, lectures, and exhibitions. Their program has been running for many decades and continues to bring diverse artists to campus. Pasadena City College: PCC has an 'Artist-in-Residence at Pasadena City College' program that brings prominent artists to campus for week-long stays. During this time, the artists interact closely with students and faculty, often including a public lecture and exhibition. They have hosted a significant number of well-known artists since the program's inception in 1987. San Joaquin Delta College: Their LH Horton Jr Art Gallery has roots in an 'Artists in Residence program' founded in 1969. While the focus may have shifted towards exhibitions and events, the historical connection indicates a past commitment to bringing artists to interact with the college community."],
    ["Which California community colleges have radio stations?", "Here are some California Community Colleges that have radio stations: San Joaquin Delta College (Stockton): Operates KWDC 93.5 FM, a fully functional, FCC-compliant, non-commercial radio station that serves as an experiential lab for Digital Media students. You can listen locally on 93.5 FM and globally online at KWDC.fm. They also have a RadioFX app where you can search for KWDC. Saddleback College (Mission Viejo): Partners with CSU Northridge to broadcast 'The SoCal Sound' on 88.5 FM. They also have a jazz-focused HD2 channel. The station is part of the Cinema/Television/Radio curriculum, where students produce content. Ohlone College (Fremont and Newark): Runs KOHL Radio 89.3 FM, a commercial-free station broadcasting a contemporary hit radio format. The station serves as a working lab for their Radio Broadcasting program."],
    ["What percentage of california community college students receive needs-based scholarships?", "It's difficult to pinpoint an exact, real-time percentage for the current academic year (2025-2026). However, based on the most recent data and trends: Key Observations and Estimates: High Financial Need: A significant portion of California Community College (CCC) students demonstrate substantial financial need. In the 2023-2024 academic year, over two-thirds of degree or transfer-intending students had high levels of financial need. Multiple Forms of Aid: Students receive various types of financial aid based on need, including: Pell Grants (Federal): In 2023-2024, approximately 24% of CCC students received Pell Grants, which are specifically for students with high financial need. Cal Grants (State): Around 9% of CCC students received Cal Grants in the same year, which also have financial need criteria along with academic requirements. California College Promise Grant (CCPG) (formerly BOG Fee Waiver): A substantial 47% received this grant, which waives enrollment fees for eligible low-income students. While not a scholarship for living expenses, it is a significant needs-based benefit. Overall Grant/Scholarship Receipt: In 2023-2024, about half of all CCC students received grants and scholarships (this figure likely includes Pell and Cal Grants but may not isolate other smaller need-based scholarships). Low Loan Usage: Only a small percentage (around 1.5% in 2023-24) of CCC students take out loans, indicating that grants and fee waivers are the primary means of financial assistance. Estimated Percentage of Students Receiving Need-Based Scholarships (Broader Definition): If we consider 'need-based scholarships' to include grants like Pell and Cal Grants, as well as the California College Promise Grant (which addresses a significant cost of attendance based on need), then the percentage of students receiving some form of need-based financial aid is likely well over half (potentially around 60-80% when combining the percentages, keeping in mind some students may receive multiple forms of aid)."],
    ["What are the recent enrollment trends for California community colleges?", "Recent enrollment trends for California Community Colleges (CCC) show a complex picture of decline followed by a significant rebound. Here's a breakdown of the key trends: Significant Pandemic-Related Decline: Following the onset of the COVID-19 pandemic, CCC enrollment experienced a substantial drop. Between Fall 2019 and Fall 2021, headcount enrollment declined by approximately 19-20%, representing over 300,000 fewer students. This decline was more pronounced in urban regions like San Diego, the Bay Area, and Greater Los Angeles. The drop appears to be driven more by students dropping out rather than a reduction in new enrollments. Enrollment declines were particularly large for American Indian or Alaska Native, Black, and Asian students. Fields of study with more male students, such as skilled trades, saw significant initial drops. Early Signs of Recovery and Recent Increases: Enrollment began to recover in Fall 2022, and this trend continued into Fall 2023. From Fall 2022 to Fall 2023, systemwide headcount increased by over 8%, with an additional 118,000 students. This was the highest growth among all higher education sectors in California. Full-time enrollment saw a significant increase of 4.6%, and dual enrollment surged by 5.2%, adding 44,000 students. Nearly all student populations experienced increases in Fall 2023, including African American (+10%), Hispanic (+9%), and students of two or more races (+8%). Enrollment also increased across age groups, notably for students 65 and older (+16%) and those 19 and younger (+10%). By Spring 2024, community colleges reported a 4.7% increase in enrollment compared to the previous year. Projections suggest a potential 9% surge in undergraduate enrollment from 2021 to 2031. Annual headcount for the 2023-2024 academic year is projected to have exceeded 2 million students for the first time since before the pandemic. There were 2.1 million students enrolled in 2023-2024. Ongoing Challenges and Context: Despite the recent growth, overall enrollment levels still remain below pre-pandemic peaks (around 11-14% lower than Fall 2019 and 2018-19). California's K-12 enrollment is projected to decline significantly in the coming years, which could pose future challenges for community college enrollment. Demographic shifts continue to play a role, with Hispanic students making up an increasing percentage of the student body. Dual enrollment is becoming an increasingly significant component of overall enrollment. In summary, while California Community Colleges experienced a substantial enrollment decline during the COVID-19 pandemic, recent trends indicate a strong rebound in both headcount and full-time enrollment. The system is on a positive trajectory toward recovering pre-pandemic enrollment levels, although demographic shifts and K-12 enrollment trends will continue to be important factors to monitor."
    ]
     ]

## Find the questions to add

In [19]:

# Get questions already in qa
existing_qs = [qa[0] for qa in qa_base if qa[0] in logs_df["question"].values]
corrected_qs = [q[0] for q in corrected_answers]
skip_qs = existing_qs + corrected_qs

# Create a list of new questions with answers to add the our Q and A reference set
qas_toadd = []
for idx in logs_df.index[:30]:
    if logs_df.loc[idx, "question"] not in skip_qs:
        qas_toadd.append([logs_df.loc[idx, "question"],
                          logs_df.loc[idx, "response"]])


# Add all Q and A lists together
new_qas = qas_toadd + corrected_answers + qa_base



# Save to JSON

In [29]:
# Save a prettier JSON
qas_json = json.dumps(new_qas, indent=4)
with open(os.path.join(qa_data_path, "ipeds_qa_2.json"), "w") as file:
    file.write(qas_json)
